# 10b_enterocytes — Deep-Dive Differential Accessibility in Enterocytes

**Goal:** Full multi-level differential accessibility analysis focused exclusively on
**Enterocytes** — the major absorptive epithelial cell type of the small intestine.

## Analysis levels

### Level 1 — Focused DESeq2
- Re-run shared-peaks and evolutionary branch analysis on enterocytes only
- Higher stringency than all-cell-type run: `min_samples = 3`, `sd_thresh = 2.5`
- Per-contrast outlier detection + named outlier log
- Ultra-robust filtering (`min(pos) > max(neg)` in logCPM)
- Volcano plots per contrast, species marker heatmap, evolutionary staircase

### Level 2 — Motif Enrichment
- monaLisa (JASPAR2022 vertebrate PWMs) on peaks from each phylogenetic node
- Separate enrichment for: phylogenetic nodes, ILS contrasts, human-specific peaks
- Motif enrichment heatmap

### Level 3 — Gene Annotation, GO Enrichment & Visualization
- Nearest-gene annotation for all significant peaks
- GO enrichment (BP) per evolutionary contrast
- Accessibility dotplots (per-species) for top peaks at each node
- Individual strip plots for hand-picked top loci
- Per-region (Duodenum vs Colon) comparison using region-preserved data

**Requires:**
- `pb_data_shared.rds` / `pb_data_union.rds` from `10a`
- `pb_data_shared_per_region.rds` / `pb_data_union_per_region.rds` from `10a` (Cell 10)
- `master_annotation_final.tsv`

**Outputs:**
- `differential_results/enterocytes/` — per-contrast CSV + ultra-robust CSV
- `plots/Enterocytes_Deep/` — all plots for this analysis

In [ ]:
# =============================================================================
# Cell 1: Load Packages & Source Utilities
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(ggplot2)
  library(ggrepel)
  library(pheatmap)
  library(viridis)
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})

source("../src/deseq2_utils.R")
message("Packages loaded & utilities sourced.")

In [ ]:
# =============================================================================
# Cell 2: Configuration & Data Loading
# =============================================================================
BASE    <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

ENT_OUT  <- OUT_DIR  # results routed to OUT_DIR/{CellType}/ by new structure
ENT_PLOT <- file.path(OUT_DIR, "Enterocytes", "plots")
dir.create(ENT_OUT,  showWarnings = FALSE, recursive = TRUE)
dir.create(ENT_PLOT, showWarnings = FALSE, recursive = TRUE)

save_dir <- file.path(OUT_DIR, "pseudobulk")

# --- Load global pseudobulk data (aggregated, 241 samples) ---
pb_shared    <- readRDS(file.path(save_dir, "pb_data_shared.rds"))
pb_union     <- readRDS(file.path(save_dir, "pb_data_union.rds"))
counts_all   <- pb_shared$counts
meta_all     <- pb_shared$meta
counts_union <- pb_union$counts

# --- Load per-region data for Level 3 enterocyte region analysis ---
pb_shared_reg <- readRDS(file.path(save_dir, "pb_data_shared_per_region.rds"))
pb_union_reg  <- readRDS(file.path(save_dir, "pb_data_union_per_region.rds"))

# --- Subset to enterocytes ---
# The cell type name after make.names() is "Enterocytes"
ENT_CT <- "Enterocytes"
meta_ent   <- meta_all[meta_all$cell_type == ENT_CT, ]
counts_ent <- counts_all[, rownames(meta_ent)]

message("Enterocyte samples: ", nrow(meta_ent))
print(table(meta_ent$species))

# --- Load master annotation & build orthology index ---
anno_file   <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
master_anno <- read_tsv(anno_file, show_col_types = FALSE)
options(scipen = 999)

ortho_mat <- build_orthology_index(master_anno, SPECIES)
message("Orthology index: ", nrow(ortho_mat), " peaks x ", ncol(ortho_mat), " species")

---
## Level 1: Focused DESeq2 on Enterocytes

Higher stringency than the all-cell-type run: `min_samples = 3` (enterocytes are
well-represented, so this is achievable) and `sd_thresh = 2.5` for outlier detection.

In [ ]:
# =============================================================================
# Cell 3: Shared-Peak Analysis (enterocytes, all-vs-one)
# =============================================================================
# Build a meta_shared-compatible object with only enterocytes
meta_ent_factor           <- meta_ent
meta_ent_factor$cell_type <- factor(ENT_CT)
meta_ent_factor$species   <- factor(meta_ent_factor$species, levels = SPECIES)

# Re-run with higher stringency: min 3 samples per peak
res_ent_shared <- run_deseq2_shared_peaks(
  counts_shared   = counts_ent,
  meta_shared     = meta_ent_factor,
  species         = SPECIES,
  out_dir         = ENT_OUT,
  filter_outliers = TRUE,
  sd_thresh       = 2.5
)

# Summary
for (sp in names(res_ent_shared[[ENT_CT]])) {
  res_df  <- as.data.frame(res_ent_shared[[ENT_CT]][[sp]])
  sig_up  <- sum(!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1)
  sig_dn  <- sum(!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange < -1)
  message(sprintf("  %-12s: %d UP, %d DOWN", sp, sig_up, sig_dn))
}

In [ ]:
# =============================================================================
# Cell 4: Evolutionary Branch Analysis (enterocytes)
# =============================================================================
evo_contrasts <- define_evolutionary_contrasts()

branch_ent <- run_deseq2_evolutionary(
  counts_union    = counts_union,
  meta_shared     = meta_ent_factor,
  contrasts       = evo_contrasts,
  ortho_mat       = ortho_mat,
  out_dir         = ENT_OUT,
  min_samples     = 3,
  filter_outliers = TRUE,
  sd_thresh       = 2.5
)

message("\nEvolutionary branch analysis complete for enterocytes.")
message("Contrasts with results: ", length(branch_ent[[ENT_CT]]))

In [ ]:
# =============================================================================
# Cell 5: Ultra-Robust Filtering
# =============================================================================
robust_ent <- ultra_robust_filter(
  branch_res   = branch_ent,
  counts_union = counts_union,
  meta_shared  = meta_ent_factor,
  contrasts    = evo_contrasts,
  out_dir      = ENT_OUT,
  padj_thresh  = 0.05,
  lfc_thresh   = 1,
  min_logcpm   = 1
)

# Summarise ultra-robust peak counts per contrast
ur_counts <- sapply(robust_ent[[ENT_CT]], length)
ur_df     <- data.frame(
  contrast = names(ur_counts),
  n_ultra_robust = as.integer(ur_counts)
) %>% arrange(desc(n_ultra_robust))

message("\nUltra-robust peaks per contrast (enterocytes):")
print(ur_df)

In [ ]:
# =============================================================================
# Cell 6: Volcano Plots (shared-peaks + key evolutionary branches)
# =============================================================================

# A) Shared-peak volcanos (one per species)
for (sp in names(res_ent_shared[[ENT_CT]])) {
  res_df   <- as.data.frame(res_ent_shared[[ENT_CT]][[sp]])
  out_file <- file.path(ENT_PLOT, paste0("Volcano_SharedPeaks_", sp, ".pdf"))
  plot_volcano(res_df,
               title    = paste(sp, "vs All Others — Enterocytes [shared peaks]"),
               out_file = out_file, n_label = 12)
}

# B) Key evolutionary branch volcanos
key_contrasts <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo",
  "Pair_Human_vs_Macaque", "Pair_Human_vs_Marmoset"
)

for (cn in intersect(key_contrasts, names(branch_ent[[ENT_CT]]))) {
  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
  out_file <- file.path(ENT_PLOT, paste0("Volcano_Branch_", cn, ".pdf"))
  plot_volcano(res_df,
               title    = paste(cn, "— Enterocytes [evolutionary branch]"),
               out_file = out_file, n_label = 12)
}

message("All volcano plots saved.")

In [ ]:
# =============================================================================
# Cell 7: Species Marker Heatmap (shared-peak results)
# =============================================================================
# Build a temporary VST for enterocytes only (more sensitive than global VST)
dds_ent_vst <- DESeqDataSetFromMatrix(
  countData = round(counts_ent),
  colData   = meta_ent_factor,
  design    = ~ species
)
dds_ent_vst <- estimateSizeFactors(dds_ent_vst, type = "poscounts")
vsd_ent     <- vst(dds_ent_vst, blind = FALSE)
message("Enterocyte-specific VST computed.")

plot_species_heatmap(
  res_ct    = res_ent_shared[[ENT_CT]],
  vsd       = vsd_ent,
  meta      = meta_ent_factor,
  cell_type = ENT_CT,
  out_file  = file.path(ENT_PLOT, "Heatmap_Species_Markers.pdf"),
  top_n     = 30
)

In [ ]:
# =============================================================================
# Cell 8: Evolutionary Staircase Heatmap
# =============================================================================
species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
node_order    <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys", "Div_Human_vs_Apes",
  "Div_Human_vs_AllPrimates"
)

ct_nodes <- intersect(node_order, names(robust_ent[[ENT_CT]]))
ct_nodes <- ct_nodes[sapply(ct_nodes,
  function(n) length(robust_ent[[ENT_CT]][[n]]) > 0)]

# Compute logCPM averages per species
ent_samples <- rownames(meta_ent)
ent_counts_u <- counts_union[, ent_samples, drop = FALSE]
lib_sizes  <- colSums(ent_counts_u)
cpm_mat    <- t(t(ent_counts_u) / lib_sizes) * 1e6
logcpm_ent <- log2(cpm_mat + 1)

avg_exp <- matrix(NA, nrow = nrow(logcpm_ent), ncol = length(species_order),
                  dimnames = list(rownames(logcpm_ent), species_order))
for (sp in species_order) {
  sp_samp <- ent_samples[meta_ent$species == sp]
  if (length(sp_samp) > 1)
    avg_exp[, sp] <- rowMeans(logcpm_ent[, sp_samp, drop = FALSE])
  else if (length(sp_samp) == 1)
    avg_exp[, sp] <- logcpm_ent[, sp_samp]
}

plot_peaks <- c()
row_splits <- c()
for (node in ct_nodes) {
  ur_peaks <- robust_ent[[ENT_CT]][[node]]
  if (length(ur_peaks) == 0) next
  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[node]])
  ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
  sorted   <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
  top      <- head(sorted, 50)
  plot_peaks <- c(plot_peaks, top)
  row_splits <- c(row_splits, rep(node, length(top)))
}

if (length(plot_peaks) >= 5) {
  mat <- avg_exp[plot_peaks, , drop = FALSE]
  valid_cols <- apply(mat, 2, function(x) !all(is.na(x)))
  mat <- mat[, valid_cols, drop = FALSE]
  mat_scaled <- t(apply(mat, 1, scale))
  colnames(mat_scaled) <- colnames(mat)
  split_factor <- factor(row_splits, levels = ct_nodes)

  col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))
  ht <- Heatmap(mat_scaled, name = "Z-score", col = col_fun,
                row_split = split_factor, row_title_rot = 0,
                row_title_gp = gpar(fontsize = 10, fontface = "bold"),
                row_gap = unit(3, "mm"), cluster_rows = TRUE,
                cluster_columns = FALSE, show_row_names = FALSE,
                show_column_names = TRUE, column_names_rot = 45,
                column_title = "Evolutionary Staircase — Enterocytes",
                column_title_gp = gpar(fontsize = 14, fontface = "bold"),
                heatmap_legend_param = list(title = "Z-score"))

  dynamic_h <- max(10, length(plot_peaks) * 0.09 + length(ct_nodes))
  pdf(file.path(ENT_PLOT, "Evolutionary_Staircase_Enterocytes.pdf"),
      width = 10, height = dynamic_h)
  draw(ht, merge_legend = TRUE)
  dev.off()
  message("Staircase heatmap saved (", length(plot_peaks), " peaks).")
}

---
## Level 2: Motif Enrichment

Run monaLisa binned motif enrichment (JASPAR2022 vertebrate PWMs) on peak groups
defined by phylogenetic node. Each group = ultra-robust peaks for one contrast.

**Groups tested:**
1. Phylogenetic nodes (Node1–4 + Node_Apes_vs_Monkeys)
2. ILS contrasts (Human+Gorilla vs Pan; Human+Chimp vs Gorilla+Bonobo)
3. Human-specific (Div_Human_vs_Apes + Div_Human_vs_AllPrimates)

Background = all enterocyte accessible peaks (shared, 71k).

In [ ]:
# =============================================================================
# Cell 9: Motif Enrichment — Phylogenetic Node Peaks
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(JASPAR2022)
  library(TFBSTools)
  library(BSgenome.Hsapiens.UCSC.hg38)
  library(Biostrings)
})

motif_dir <- file.path(ENT_OUT, ENT_CT, "motif")
dir.create(motif_dir, showWarnings = FALSE, recursive = TRUE)

# --- Build peak groups for motif enrichment ---
# Use ultra-robust peaks; label each by the contrast it came from.
# Only keep groups with >= 20 peaks (minimum for monaLisa to be stable).
motif_groups <- c(
  # Phylogenetic nodes
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  # ILS
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo",
  # Human-specific
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates"
)

peak_list_motif <- list()
for (cn in intersect(motif_groups, names(robust_ent[[ENT_CT]]))) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) >= 20) {
    peak_list_motif[[cn]] <- peaks
    message(sprintf("  %-40s: %d peaks", cn, length(peaks)))
  } else {
    message(sprintf("  %-40s: %d peaks — SKIPPED (< 20)", cn, length(peaks)))
  }
}

if (length(peak_list_motif) == 0) {
  message("No groups with >= 20 peaks. Adjust thresholds or run DESeq2 with looser settings.")
} else {
  message("Running motif enrichment for ", length(peak_list_motif), " groups...")

  se_motif_nodes <- run_parallel_motif_enrichment(
    peak_list_named = peak_list_motif,
    anno_df         = master_anno,
    species         = "Human",
    n_cores         = 4L,
    out_rds         = file.path(motif_dir, "se_motif_phylo_nodes.rds")
  )

  plot_motif_heatmap(
    se_obj      = se_motif_nodes,
    title       = "Motif Enrichment — Enterocyte Phylogenetic Node Peaks",
    out_file    = file.path(ENT_PLOT, "Motif_Heatmap_PhyloNodes.pdf"),
    top_per_col = 5
  )
  message("Phylogenetic node motif enrichment complete.")
}

In [ ]:
# =============================================================================
# Cell 10: Motif Enrichment — ILS Contrast Peaks
# =============================================================================
ils_groups <- c(
  "ILS_HumanGorilla_vs_Pan",
  "ILS_HumanChimp_vs_GorillaBonobo",
  "ILS_HumanBonobo_vs_ChimpGorilla"
)

peak_list_ils <- list()
for (cn in intersect(ils_groups, names(robust_ent[[ENT_CT]]))) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) >= 20) peak_list_ils[[cn]] <- peaks
}

if (length(peak_list_ils) >= 2) {
  se_motif_ils <- run_parallel_motif_enrichment(
    peak_list_named = peak_list_ils,
    anno_df         = master_anno,
    species         = "Human",
    n_cores         = 4L,
    out_rds         = file.path(motif_dir, "se_motif_ILS.rds")
  )
  plot_motif_heatmap(
    se_obj      = se_motif_ils,
    title       = "Motif Enrichment — Enterocyte ILS Peaks",
    out_file    = file.path(ENT_PLOT, "Motif_Heatmap_ILS.pdf"),
    top_per_col = 5
  )
  message("ILS motif enrichment complete.")
} else {
  message("Insufficient ILS groups with >= 20 peaks. Skipping.")
}

---
## Level 3: Gene Annotation, GO Enrichment & Visualization

In [ ]:
# =============================================================================
# Cell 11: Gene Annotation for All Significant Peaks
# =============================================================================
gene_anno_dir <- file.path(ENT_OUT, ENT_CT, "gene_annotation")
dir.create(gene_anno_dir, showWarnings = FALSE, recursive = TRUE)

# Collect all ultra-robust peaks with their contrast labels
all_ur_peaks <- lapply(names(robust_ent[[ENT_CT]]), function(cn) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) == 0) return(NULL)
  data.frame(peak_id = peaks, contrast = cn, stringsAsFactors = FALSE)
})
all_ur_df <- do.call(rbind, Filter(Negate(is.null), all_ur_peaks))

# Add gene annotation from master_anno
gene_info <- get_peak_info(all_ur_df$peak_id, "Human", "gene", master_anno)
dist_info <- get_peak_info(all_ur_df$peak_id, "Human", "distance", master_anno)

all_ur_annotated <- all_ur_df %>%
  left_join(gene_info,     by = "peak_id") %>%
  left_join(dist_info,     by = "peak_id") %>%
  left_join(
    do.call(rbind, lapply(names(branch_ent[[ENT_CT]]), function(cn) {
      res_df <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
      data.frame(peak_id = rownames(res_df), contrast = cn,
                 log2FC = round(res_df$log2FoldChange, 3),
                 padj   = signif(res_df$padj, 4),
                 stringsAsFactors = FALSE)
    })),
    by = c("peak_id", "contrast")
  )

write.csv(all_ur_annotated,
          file.path(gene_anno_dir, "Enterocytes_UltraRobust_Annotated.csv"),
          row.names = FALSE)

message("Annotated ultra-robust peaks: ", nrow(all_ur_annotated))
message("Unique genes: ", length(unique(all_ur_annotated$gene[all_ur_annotated$gene != "."])))
head(all_ur_annotated[all_ur_annotated$contrast == "Div_Human_vs_Apes", ], 10)

In [ ]:
# =============================================================================
# Cell 12: GO + Disease Enrichment per Evolutionary Contrast
# =============================================================================
# run_full_enrichment() calls both run_go_enrichment() and run_disease_enrichment()
# (KEGG, Disease Ontology, DisGeNET, Cancer Genes) for each contrast.
# Universe = all peaks tested per contrast (from branch_ent), not the whole genome.
enrich_contrasts <- c(
  # Phylogenetic nodes
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  # Human divergence
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  # ILS contrasts
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo",
  "ILS_HumanBonobo_vs_ChimpGorilla"
)

# Also run on species-specific shared-peak results (UP direction)
species_enrich <- c("Human", "Marmoset", "Macaque", "Gorilla")

message("=== GO + Disease Enrichment: Evolutionary Contrasts ===")
for (cn in intersect(enrich_contrasts, names(robust_ent[[ENT_CT]]))) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  # Universe: all peaks tested for this contrast (from branch_ent)
  uni_peaks <- if (!is.null(branch_ent[[ENT_CT]][[cn]]))
    rownames(as.data.frame(branch_ent[[ENT_CT]][[cn]])) else NULL
  run_full_enrichment(
    peaks          = peaks,
    species        = "Human",
    label          = paste0("Ent_", cn),
    out_dir        = ENT_OUT,
    anno_df        = master_anno,
    universe_peaks = uni_peaks
  )
}

message("\n=== GO + Disease Enrichment: Species-Specific Peaks ===")
if (!is.null(res_ent_shared[[ENT_CT]])) {
  for (sp in species_enrich) {
    sp_res <- res_ent_shared[[ENT_CT]][[sp]]
    if (is.null(sp_res)) next
    df     <- as.data.frame(sp_res)
    peaks_up <- rownames(df[!is.na(df$padj) & df$padj < 0.05 & df$log2FoldChange > 1, ])
    run_full_enrichment(
      peaks          = peaks_up,
      species        = "Human",
      label          = paste0("Ent_SpeciesSpecific_", sp),
      out_dir        = ENT_OUT,
      anno_df        = master_anno,
      ct             = ENT_CT,
      universe_peaks = rownames(df)
    )
  }
}

message("\nFull enrichment complete. Results in: ",
        file.path(ENT_OUT, ENT_CT))
message("  GO:      differential_results/enrichment/")
message("  Disease: differential_results/disease_enrichment/")


In [ ]:
# =============================================================================
# Cell 13: Accessibility Dotplots — Top Peaks per Node
# =============================================================================
# For each evolutionary contrast, plot top 40 ultra-robust peaks as a
# per-species dotplot (mean log2CPM, colored by accessibility level)

dotplot_contrasts <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo"
)

ent_samples <- rownames(meta_ent)

for (cn in intersect(dotplot_contrasts, names(robust_ent[[ENT_CT]]))) {
  ur_peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(ur_peaks) == 0) next

  # Sort by LFC to show highest-contrast peaks first
  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
  ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
  sorted   <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
  top_peaks <- head(sorted, 40)

  tryCatch({
    plot_accessibility_dotplot(
      peak_ids     = top_peaks,
      counts       = counts_union[, ent_samples, drop = FALSE],
      meta         = meta_ent,
      out_file     = file.path(ENT_PLOT, paste0("Dotplot_", cn, ".pdf")),
      title        = paste("Enterocytes:", cn, "— Ultra-Robust Peaks"),
      max_peaks    = 40
    )
  }, error = function(e) message("  Dotplot failed for ", cn, ": ", e$message))
}

message("All accessibility dotplots saved.")

In [ ]:
# =============================================================================
# Cell 14: Strip Plots — Individual Top Loci
# =============================================================================
# For each major contrast, make per-donor strip plots for the 5 highest-LFC
# ultra-robust peaks. Shows raw variability across donors — key for sanity-checking.

strip_contrasts <- c(
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "Node1_Human_vs_Pan", "Node_Apes_vs_Monkeys"
)

strip_dir <- file.path(ENT_PLOT, "StripPlots_TopLoci")
dir.create(strip_dir, showWarnings = FALSE, recursive = TRUE)

for (cn in intersect(strip_contrasts, names(robust_ent[[ENT_CT]]))) {
  ur_peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(ur_peaks) == 0) next

  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
  ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
  top5     <- head(rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)], 5)

  # Also get gene annotation for labeling
  gene_lut <- setNames(
    get_peak_info(top5, "Human", "gene", master_anno)$gene,
    get_peak_info(top5, "Human", "gene", master_anno)$peak_id
  )

  for (pk in top5) {
    gene_label <- gene_lut[pk]
    title_str  <- if (!is.na(gene_label) && gene_label != ".") {
      paste0(pk, "  (", gene_label, ")  |", cn)
    } else {
      paste0(pk, "  |", cn)
    }
    out_file <- file.path(strip_dir,
                          paste0(cn, "_", gsub(":", "_", pk), ".pdf"))
    tryCatch(
      plot_peak_strip(pk,
                      counts   = counts_union[, ent_samples, drop = FALSE],
                      meta     = meta_ent,
                      out_file = out_file,
                      title    = title_str),
      error = function(e) message("  Strip plot failed: ", pk)
    )
  }
  message("  Strip plots saved for: ", cn)
}

message("\nAll strip plots saved.")

## Level 3b: Per-Region Analysis (Duodenum vs Colon)

Run the full evolutionary contrast set on enterocytes within Duodenum and Colon
separately, then compare. Enterocytes are more abundant in duodenum (absorptive);
expect more peaks there, but some regulatory elements may be colon-restricted.

In [ ]:
# =============================================================================
# Cell 15: Per-Region DESeq2 for Enterocytes
# =============================================================================
TARGET_REGIONS <- c("Duodenum", "Colon")

meta_ent_reg    <- pb_shared_reg$meta[pb_shared_reg$meta$cell_type == ENT_CT, ]
counts_ent_u_reg <- pb_union_reg$counts[,
  intersect(rownames(meta_ent_reg), colnames(pb_union_reg$counts)),
  drop = FALSE]
meta_ent_reg <- meta_ent_reg[
  intersect(rownames(meta_ent_reg), colnames(counts_ent_u_reg)), ]

message("Per-region enterocyte samples:")
print(table(meta_ent_reg$region, meta_ent_reg$species))

region_res_ent <- run_deseq2_per_region(
  counts_union_regions  = counts_ent_u_reg,
  meta_regions          = meta_ent_reg,
  contrasts             = evo_contrasts,
  ortho_mat             = ortho_mat,
  out_dir               = ENT_OUT,
  target_regions        = TARGET_REGIONS,
  min_samples           = 2,
  min_cells_region      = 30,
  min_counts_region     = 20000,
  filter_outliers       = TRUE,
  sd_thresh             = 2.5
)

message("Per-region enterocyte DESeq2 complete.")

In [ ]:
# =============================================================================
# Cell 16: Region Comparison for Enterocytes
# =============================================================================
comp_dir_ent <- file.path(ENT_OUT, ENT_CT, "region_comparison")
dir.create(comp_dir_ent, showWarnings = FALSE, recursive = TRUE)

key_contrasts_reg <- c(
  "Node1_Human_vs_Pan", "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys"
)

summary_reg_rows <- list()

for (cn in key_contrasts_reg) {
  comp <- compare_regions(region_res_ent, ENT_CT, cn,
                          padj_thresh = 0.05, lfc_thresh = 1)
  if (is.null(comp) || nrow(comp) == 0) {
    message("  No significant peaks for: ", cn)
    next
  }
  write.csv(comp, file.path(comp_dir_ent, paste0(cn, "_region_comparison.csv")),
            row.names = FALSE)

  n_both  <- sum(comp$category == "Both")
  n_duod  <- sum(comp$category == "Duodenum_only")
  n_colon <- sum(comp$category == "Colon_only")
  message(sprintf("  %-40s Both=%d  Duod=%d  Colon=%d",
                  cn, n_both, n_duod, n_colon))

  summary_reg_rows[[cn]] <- data.frame(
    contrast = cn, n_both = n_both,
    n_duodenum_only = n_duod, n_colon_only = n_colon,
    pct_conserved = round(100 * n_both / max(1, n_both + n_duod + n_colon), 1)
  )
}

if (length(summary_reg_rows) > 0) {
  reg_summary_df <- do.call(rbind, summary_reg_rows)
  write.csv(reg_summary_df,
            file.path(comp_dir_ent, "Enterocytes_Region_Comparison_Summary.csv"),
            row.names = FALSE)
  message("\nEnterocyte region comparison summary:")
  print(reg_summary_df)
}

In [ ]:
# =============================================================================
# Cell 17: Per-Region Accessibility Dotplots for Enterocytes
# =============================================================================
# For each region, show the top human-specific peaks as a dotplot,
# so we can visually compare Duodenum vs Colon regulatory landscapes.

for (region in TARGET_REGIONS) {
  if (is.null(region_res_ent[[region]])) next
  if (is.null(region_res_ent[[region]][[ENT_CT]])) next

  # Focus on human divergence contrast
  cn <- "Div_Human_vs_AllPrimates"
  res_r <- region_res_ent[[region]][[ENT_CT]][[cn]]
  if (is.null(res_r)) next

  res_df_r <- as.data.frame(res_r)
  top_peaks_r <- rownames(res_df_r)[
    !is.na(res_df_r$padj) & res_df_r$padj < 0.05 &
    res_df_r$log2FoldChange > 1
  ]
  top_peaks_r <- head(
    top_peaks_r[order(res_df_r[top_peaks_r, "log2FoldChange"], decreasing = TRUE)],
    30
  )
  if (length(top_peaks_r) < 5) next

  reg_samp <- rownames(meta_ent_reg)[meta_ent_reg$region == region]
  tryCatch({
    plot_accessibility_dotplot(
      peak_ids  = top_peaks_r,
      counts    = counts_ent_u_reg[, reg_samp, drop = FALSE],
      meta      = meta_ent_reg[reg_samp, ],
      out_file  = file.path(ENT_PLOT,
                            paste0("Dotplot_", region, "_HumanSpecific.pdf")),
      title     = paste("Enterocytes –", region, "– Human-specific peaks")
    )
  }, error = function(e) message("  Dotplot failed for ", region, ": ", e$message))
}

message("Per-region enterocyte dotplots saved.")

In [ ]:
# =============================================================================
# Cell 18: Region Comparison Heatmap for Enterocytes
# =============================================================================
tryCatch({
  plot_region_comparison_heatmap(
    region_res = region_res_ent,
    cell_type  = ENT_CT,
    out_file   = file.path(ENT_PLOT, "Region_Comparison_Heatmap_Enterocytes.pdf")
  )
}, error = function(e) {
  message("Region comparison heatmap failed: ", e$message)
})

In [ ]:
# =============================================================================
# Cell 19: Final Checkpoint & Output Summary
# =============================================================================
message("\n========== Enterocyte Deep-Dive Complete ==========")
message()
message("=== Level 1 — Focused DESeq2 ===")
message("  Shared-peak contrasts (all-vs-one): ", length(res_ent_shared[[ENT_CT]]))
message("  Evolutionary branch contrasts:      ", length(branch_ent[[ENT_CT]]))
total_ur <- sum(sapply(robust_ent[[ENT_CT]], length))
message("  Ultra-robust peaks (total):         ", total_ur)

message()
message("=== Level 2 — Motif Enrichment ===")
message("  Results in: ", file.path(ENT_OUT, "motif_enrichment"))

message()
message("=== Level 3 — Gene Annotation & Visualization ===")
message("  Annotated peaks: ",
        file.path(ENT_OUT, "gene_annotation", "Enterocytes_UltraRobust_Annotated.csv"))
message("  Enrichment:  ", file.path(ENT_OUT, ENT_CT, "enrichment"))
message("  All plots:       ", ENT_PLOT)

message()
message("=== Per-Region ===")
for (rg in TARGET_REGIONS) {
  if (!is.null(region_res_ent[[rg]])) {
    n_c <- length(region_res_ent[[rg]][[ENT_CT]])
    message(sprintf("  %-12s: %d contrast results", rg, n_c))
  }
}